In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from shapely.geometry import box
from matplotlib.patches import Patch


pd.options.display.max_colwidth = 100
pd.options.display.max_rows = 10
pd.options.display.max_columns = 30

In [ ]:
# read in data

# evac data: dissolve to one multipolygon because we dont care about warning/order differentiating
evac_palisades = gpd.read_parquet('../01_data/02_clean/evac_palisades.parquet')
evac_palisades = evac_palisades[['geometry']]
evac_palisades = evac_palisades.dissolve()

evac_eaton = gpd.read_parquet('../01_data/02_clean/evac_eaton.parquet')
evac_eaton = evac_eaton[['geometry']]
evac_eaton = evac_eaton.dissolve() 

evac_crs = evac_eaton.crs # both crs's are the same
evac_union = gpd.GeoDataFrame(geometry=evac_palisades.geometry.union(evac_eaton.geometry), crs=evac_crs) # for plotting


# fires
fires = gpd.read_file('../01_data/01_raw/data_2025_01_17.geojson').to_crs(epsg=2229)
fires["poly_DateCurrent"] = fires["poly_DateCurrent"].dt.tz_convert('US/Pacific')
fires = fires[fires['poly_DateCurrent'] > '2025-01-06']
fires["poly_DateCurrent"] = fires["poly_DateCurrent"].dt.date
fires = fires[['geometry']]
fires_union = fires.dissolve()  # dissolve to one multipolygon
fires_union = fires_union.to_crs(evac_crs)  # convert to same CRS as evac data

# CA state boundary for trimming coastal CT
states = gpd.read_file('../01_data/01_raw/cb_2018_us_state_500k.shp')
ca_state = states[states['STUSPS'] == 'CA'].reset_index(drop=True)
ca_state = ca_state[['geometry']]
ca_state = ca_state.to_crs(evac_crs)

# census tracts: just need to send data to kpsc with ct geoid
cts = gpd.read_file('../01_data/01_raw/tl_2010_06_tract10.shp')
cts = cts[['GEOID10', 'geometry']].rename(columns={'GEOID10': 'geoid'})
cts = cts.to_crs(evac_crs)
cts = gpd.overlay(cts, ca_state, how='intersection') # intersect with CA state boundary
counties_kpsc = ['06037', '06111', '06083', '06079', '06029', '06059', '06073', '06025', '06065', '06071']
cts = cts[cts['geoid'].str.startswith(tuple(counties_kpsc))].reset_index(drop=True)  # filter by counties

# pop data path (only going to read in as needed, so doing so below)
ghs_path = '../01_data/01_raw/GHS_POP_E2025_GLOBE_R2023A_54009_100_V1_0/GHS_POP_E2025_GLOBE_R2023A_54009_100_V1_0.tif'

In [ ]:
## The plan

# secondary analysis exposure data 
    # evac_palisades 0/1
    # evac_eaton 0/1

# Final dataset cols:
    # CT geoid
    # exposure_cat
    # secondary_exposure_cat_palisades
    # secondary_exposure_cat_eaton

In [ ]:
# census tracts exposed to eaton and palisades evac zones 
exposed_palisades_cts = cts[cts.overlaps(evac_palisades.unary_union) | cts.within(evac_palisades.unary_union)]
exposed_eaton_cts = cts[cts.overlaps(evac_eaton.unary_union) | cts.within(evac_eaton.unary_union)]

# create exposure categories
exposed_palisades_cts['evac_palisades'] = 1
exposed_eaton_cts['evac_eaton'] = 1

# compile final dataset and include all cts
final_exposure_data = cts[['geoid']].copy()
final_exposure_data = final_exposure_data.merge(exposed_palisades_cts[['geoid', 'evac_palisades']], on='geoid', how='left')
final_exposure_data = final_exposure_data.merge(exposed_eaton_cts[['geoid', 'evac_eaton']], on='geoid', how='left')

final_exposure_data['evac_palisades'] = final_exposure_data['evac_palisades'].fillna(0)
final_exposure_data['evac_palisades'] = final_exposure_data['evac_palisades'].astype(int)
final_exposure_data['evac_eaton'] = final_exposure_data['evac_eaton'].fillna(0)
final_exposure_data['evac_eaton'] = final_exposure_data['evac_eaton'].astype(int)

# write out data
final_exposure_data.to_csv('../01_data/02_clean/evac_exp_secondary.csv', index=False)

In [ ]:
# convert all geometries to Web Mercator for plotting
cts_mercator = cts.to_crs('EPSG:3857')
exposed_cts = final_exposure_data.merge(cts_mercator[['geoid', 'geometry']], on='geoid', how='left')
exposed_cts = gpd.GeoDataFrame(exposed_cts, geometry='geometry', crs='EPSG:3857')
fires_mercator = fires_union.to_crs('EPSG:3857')

# filter evacuation zones by category
evac_palisades = exposed_cts[exposed_cts['evac_palisades'] == 1]
evac_eaton = exposed_cts[exposed_cts['evac_eaton'] == 1]

# calculate exposure counts
palisades_count = final_exposure_data['evac_palisades'].sum()
eaton_count = final_exposure_data['evac_eaton'].sum()

fig, ax = plt.subplots(1, 1, figsize=(15, 12))

# create bounds based on exposed census tracts area
combined_bounds_geom = exposed_cts.union_all()

# get bounds for clipping and add some buffer
minx, miny, maxx, maxy = combined_bounds_geom.bounds
buffer_x = (maxx - minx) * 0.1  # 10% buffer
buffer_y = (maxy - miny) * 0.1
clip_bounds = [minx - buffer_x, miny - buffer_y, maxx + buffer_x, maxy + buffer_y]

# clip all census tracts to the area of interest
clip_box = box(*clip_bounds)
cts_clipped = cts_mercator[cts_mercator.intersects(clip_box)]

# plot evacuation zones for each category with different colors
evac_palisades.plot(ax=ax, color='#29505d', edgecolor='none', alpha = 0.8, label='Palisades Evacuation Zone')
evac_eaton.plot(ax=ax, color='#4a6741', edgecolor='none', alpha = 0.8, label='Eaton Evacuation Zone')

# plot fire boundaries (red outlines only)
fires_mercator.boundary.plot(ax=ax, color='#8f1402', linewidth=2, alpha=0.8)

# set the axis limits to the clipped bounds of the kpsc catchment area
ax.set_xlim(clip_bounds[0], clip_bounds[2])
ax.set_ylim(clip_bounds[1], clip_bounds[3])
    
ctx.add_basemap(ax, crs='EPSG:3857', source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.6)

ax.set_title('Evacuation boundary exposure by fire', fontsize=24)

legend_elements = [
    Patch(facecolor='#29505d', alpha=0.8, label='Palisades evacuation zone'),
    Patch(facecolor='#4a6741', alpha=0.8, label='Eaton evacuation zone'),
    Patch(facecolor='none', edgecolor='#8f1402', linewidth=2, label='Fire boundaries')
]
ax.legend(handles=legend_elements, loc='upper right', fontsize = 16)

ax.set_xticks([])
ax.set_yticks([])

# add exposure counts text at the bottom of the plot
counts_text = f'Census Tracts Exposed - Palisades: {palisades_count:,} | Eaton: {eaton_count:,}'
ax.text(0.5, 0.02, counts_text, 
        transform=ax.transAxes, 
        ha='center', va='bottom',
        fontsize=16,
        bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8, edgecolor='gray'))

plt.tight_layout()
plt.show()

In [ ]:
final_exposure_data

In [ ]:
evac_eaton

In [ ]:
# convert all geometries to Web Mercator for plotting
cts_mercator = cts.to_crs('EPSG:3857')
exposed_cts = final_exposure_data.merge(cts_mercator[['geoid', 'geometry']], on='geoid', how='left')
exposed_cts = gpd.GeoDataFrame(exposed_cts, geometry='geometry', crs='EPSG:3857')
fires_mercator = fires_union.to_crs('EPSG:3857')

# filter evacuation zones by category
evac_palisades = exposed_cts[exposed_cts['evac_palisades'] == 1]
evac_eaton = exposed_cts[exposed_cts['evac_eaton'] == 1]

# calculate exposure counts
palisades_count = final_exposure_data['evac_palisades'].sum()
eaton_count = final_exposure_data['evac_eaton'].sum()

fig, ax = plt.subplots(1, 1, figsize=(15, 12))

# create bounds based on exposed census tracts area
combined_bounds_geom = exposed_cts.union_all()

# get bounds for clipping and add some buffer
minx, miny, maxx, maxy = combined_bounds_geom.bounds
buffer_x = (maxx - minx) * 0.1  # 10% buffer
buffer_y = (maxy - miny) * 0.1
clip_bounds = [minx - buffer_x, miny - buffer_y, maxx + buffer_x, maxy + buffer_y]

# clip all census tracts to the area of interest
clip_box = box(*clip_bounds)
cts_clipped = cts_mercator[cts_mercator.intersects(clip_box)]

# plot evacuation zones with colors from the second figure
# Using teal/blue colors from the exposure category plot
evac_palisades.plot(ax=ax, color='#2E8B8B', edgecolor='none', alpha=0.8, 
                   label='Palisades Evacuation Zone')
evac_eaton.plot(ax=ax, color='#0C5985', edgecolor='none', alpha=0.8, 
               label='Eaton Evacuation Zone')

# plot fire boundaries using the red from the second figure
fires_mercator.boundary.plot(ax=ax, color='#9A3334', linewidth=2, alpha=0.8)

# set the axis limits to the clipped bounds
ax.set_xlim(clip_bounds[0], clip_bounds[2])
ax.set_ylim(clip_bounds[1], clip_bounds[3])

ctx.add_basemap(ax, crs='EPSG:3857', source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.6)

ax.set_title('Evacuation boundary exposure by fire', fontsize=24)

legend_elements = [
    Patch(facecolor='#2E8B8B', alpha=0.8, label='Palisades evacuation zone'),
    Patch(facecolor='#0C5985', alpha=0.8, label='Eaton evacuation zone'),
    Patch(facecolor='none', edgecolor='#9A3334', linewidth=2, label='Fire boundaries')
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=16)

ax.set_xticks([])
ax.set_yticks([])

# add exposure counts text at the bottom of the plot
counts_text = f'Census Tracts Exposed - Palisades: {palisades_count:,} | Eaton: {eaton_count:,}'
ax.text(0.5, 0.02, counts_text,
        transform=ax.transAxes,
        ha='center', va='bottom',
        fontsize=16,
        bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8, edgecolor='gray'))

plt.tight_layout()
plt.show()

In [ ]:
exposed_cts

In [ ]:
# Define exposed census tracts as those with either evac_eaton == 1 OR evac_palisades == 1
exposed_ids = final_exposure_data[(final_exposure_data['evac_eaton'] == 1) | 
                                  (final_exposure_data['evac_palisades'] == 1)]

# Merge with the census tracts geodataframe to get geometries
exposed_cts = cts.merge(exposed_ids[['geoid']], on='geoid', how='inner')

# Convert all geometries to Web Mercator for plotting
cts_mercator = cts.to_crs('EPSG:3857')
exposed_cts_mercator = exposed_cts.to_crs('EPSG:3857')
evac_mercator = evac_union.to_crs('EPSG:3857')  # Single evacuation zone polygon
fires_mercator = fires_union.to_crs('EPSG:3857')

fig, ax = plt.subplots(1, 1, figsize=(15, 12))

# Create bounds based on exposed census tracts area
combined_bounds_geom = exposed_cts_mercator.union_all()

# Get bounds for clipping and add some buffer
minx, miny, maxx, maxy = combined_bounds_geom.bounds
buffer_x = (maxx - minx) * 0.1  # 10% buffer
buffer_y = (maxy - miny) * 0.1
clip_bounds = [minx - buffer_x, miny - buffer_y, maxx + buffer_x, maxy + buffer_y]

# Clip all census tracts to the area of interest
clip_box = box(*clip_bounds)
cts_clipped = cts_mercator[cts_mercator.intersects(clip_box)]

# Plot all census tracts (outline only, no fill)
cts_clipped.plot(ax=ax, facecolor='none', edgecolor='gray', linewidth=0.5, alpha=0.7)

# Plot intersecting census tracts in blue (those with evac_eaton == 1 OR evac_palisades == 1)
exposed_cts_mercator.plot(ax=ax, color='#0C5985', edgecolor='none', linewidth=1)

# Plot evacuation zones as single polygon
evac_mercator.plot(ax=ax, edgecolor='#66A8CF', color='none', linewidth=2)

# Plot fire boundaries (red outlines only)
fires_mercator.boundary.plot(ax=ax, color='#8f1402', linewidth=2, alpha=0.8)

# Set the axis limits to the clipped bounds
ax.set_xlim(clip_bounds[0], clip_bounds[2])
ax.set_ylim(clip_bounds[1], clip_bounds[3])

# Create legend
legend_elements = [
    Patch(facecolor='none', edgecolor='gray', linewidth=0.5, label='Census tracts'),
    Patch(facecolor='#0C5985', label='Exposed census tracts (Eaton or Palisades)'),
    Patch(facecolor='none', edgecolor='#66A8CF', linewidth=2, label='Evacuation zone'),
    Patch(facecolor='none', edgecolor='#8f1402', linewidth=2, label='Fire Boundaries')
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=16)

ax.set_xticks([])
ax.set_yticks([])

plt.tight_layout()
plt.show()